# How Thick Are New York City's Street Trees?

*Taking the measure of 680,000 trees with the mean, the median, the mode, and a chart made of emoji.*

Walk down almost any block in New York City and you'll pass a street tree: a scrawny sapling held up by a stake, or a huge London planetree whose roots have cracked the sidewalk. About once every ten years, NYC Parks and thousands of volunteers try to count **every single one** of them. This is the [TreesCount! Street Tree Census](https://www.nycgovparks.org/trees/treescount).

Volunteers in the 2015 census wrapped a tape measure around each trunk and recorded its thickness. That gives me a question:

> **How big is a "typical" New York City street tree?**

"Typical" can mean different things, so I'll answer it three ways:

- **Mean**: add up every trunk and divide by the number of trees (the everyday "average").
- **Median**: line the trees up from thinnest to thickest and pick the one in the middle.
- **Mode**: the single most common trunk size.

I'll calculate each one twice. First I'll use [pandas](https://pandas.pydata.org/), a popular data analysis library. Then I'll do it "the hard way," with only the tools built into Python. Finally, I'll draw a chart of the trunk sizes using nothing but text.

## The data

I'm using the **[2015 Street Tree Census - Tree Data](https://data.cityofnewyork.us/Environment/2015-Street-Tree-Census-Tree-Data/uvpi-gqnh)** from NYC Open Data, published by the NYC Department of Parks & Recreation. It's a primary source: the raw records collected during the census. Each row is one tree (or stump) on a city street, and there are roughly 680,000 rows.

The dataset page includes a data dictionary (under *About → Attachments*). It explains the column I care about:

- **`tree_dbh`**: the tree's **d**iameter at **b**reast **h**eight, in inches. This is measured about 4.5 feet (54 inches) above the ground, the standard way foresters measure tree size. It's recorded in whole inches.

The full dataset has 45 columns, most of which I don't need, like addresses, council districts, and whether the roots are tangled in wires. NYC Open Data has an API that lets me ask for just a few columns, which makes the download much smaller and faster. The URL below asks for five columns:

- `tree_id`: a unique ID for each tree
- `tree_dbh`: the trunk diameter (**my numeric column**)
- `status`: whether the tree is `Alive`, `Dead`, or a `Stump`
- `spc_common`: the species' common name
- `boroname`: the borough

The API only returns 1,000 rows unless you tell it otherwise, so I also set a `$limit` that's higher than the number of rows in the dataset. The download takes about a minute, so the code saves it to a file and skips the download if that file already exists.

In [1]:
import math
import os
import urllib.request

DATA_URL = (
    "https://data.cityofnewyork.us/resource/uvpi-gqnh.csv"
    "?$select=tree_id,tree_dbh,status,spc_common,boroname"
    "&$limit=1000000"
)
DATA_FILE = "street_trees_2015.csv"

if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

## Part 1: The easy way, with pandas

First, I'll load the file into pandas and take a look at the first few rows.

In [2]:
import pandas as pd

trees = pd.read_csv(DATA_FILE)
trees.head()

,tree_id,tree_dbh,status,spc_common,boroname
0,180683,3,Alive,red maple,Queens
1,200540,21,Alive,pin oak,Queens
2,204026,3,Alive,honeylocust,Brooklyn
3,204337,10,Alive,honeylocust,Brooklyn
4,189565,21,Alive,American linden,Brooklyn


In [3]:
print(f"The census recorded {len(trees):,} trees and stumps.")

The census recorded 683,788 trees and stumps.


### A first look at trunk sizes

Before calculating anything, I'll use `describe()` to get a quick summary of the `tree_dbh` column.

In [4]:
trees["tree_dbh"].describe()

count    683788.000000
mean         11.279787
std           8.723042
min           0.000000
25%           4.000000
50%           9.000000
75%          16.000000
max         450.000000
Name: tree_dbh, dtype: float64

Two things stand out:

1. **The minimum is 0 inches.** A tree with no trunk?
2. **The maximum is huge.** A trunk hundreds of inches across would be dozens of feet thick. I'll come back to this one.

### A detour: the zero-inch "trees"

Here's the `status` of every row where the diameter is 0:

In [5]:
zero_diameter = trees[trees["tree_dbh"] == 0]
zero_diameter["status"].value_counts()

status
Stump    17654
Alive      222
Dead        56
Name: count, dtype: int64

Most of them are **stumps**. The data dictionary says to use a different column, `stump_diam` (which I didn't download), for a stump's size, and every stump has a `tree_dbh` of 0. So these zeros mean "no trunk to measure," not "a very thin tree."

If I kept the stumps, they'd pull the mean and median toward zero and make NYC's trees look smaller than they are. Here's the breakdown of all rows by status:

In [6]:
trees["status"].value_counts()

status
Alive    652173
Stump     17654
Dead      13961
Name: count, dtype: int64

That changes my research question a little. I'm really asking about **living trees**, so from here on I'll only use rows where `status` is `Alive`. Dead trees are still standing, but they won't grow any bigger, so I'll leave them out too.

In [7]:
living_trees = trees[trees["status"] == "Alive"]
living_diameters = living_trees["tree_dbh"]

print(f"{len(living_diameters):,} of the {len(trees):,} rows are living trees.")

652,173 of the 683,788 rows are living trees.


### Mean, median, and mode

pandas has a method for each statistic. One quirk: `.mode()` returns a *list* of values rather than a single number, because two or more values could tie for "most common." I convert it to a plain Python list and print every value in it.

In [8]:
mean_pandas = living_diameters.mean()
median_pandas = living_diameters.median()
modes_pandas = living_diameters.mode().tolist()

print(f"Mean:   {mean_pandas:.2f} inches")
print(f"Median: {median_pandas:g} inches")
print(f"Mode:   {', '.join(str(mode) for mode in modes_pandas)} inches")

Mean:   11.71 inches
Median: 10 inches
Mode:   4 inches


**What this tells us:** the three "typical" values don't agree, and how they disagree is the interesting part.

- The **mode** is the smallest of the three. The most common street tree is a skinny one, probably a young tree that was planted recently. NYC planted a *lot* of street trees in the years before this census through the [MillionTreesNYC](https://www.milliontreesnyc.org/) campaign, which may explain why.
- The **mean** is higher than the **median**. That happens when a small number of very large values pull the average up. A few giant old trees raise the mean, but they can't move the "middle tree" very far.

So if someone asks me how big a typical NYC street tree is, I'd quote the median. It describes the tree in the middle of the line-up and isn't thrown off by the giants.

### Another detour: the giants

Remember that huge maximum? Here are the five thickest living trees in the data:

In [9]:
living_trees.nlargest(5, "tree_dbh")

,tree_id,tree_dbh,status,spc_common,boroname
2405,168583,425,Alive,swamp white oak,Brooklyn
180962,370606,389,Alive,tulip-poplar,Queens
201195,418433,318,Alive,London planetree,Brooklyn
270493,476317,318,Alive,pin oak,Manhattan
227057,409369,298,Alive,Callery pear,Staten Island


A trunk 300 to 400+ inches across would be 25 to 35 feet thick. That's about the size of California's giant sequoias, the most massive trees on Earth, not a sidewalk tree in Brooklyn. These are almost certainly **data entry mistakes**. Maybe someone typed the circumference instead of the diameter, or hit an extra key.

I thought about deleting them, but I couldn't find a non-arbitrary place to draw the line between "big tree" and "typo," so I kept them. This is another reason to prefer the median: a handful of typos barely changes it. They nudge the mean up slightly, and they don't affect the mode at all.

## Part 2: The hard way, with only standard Python

Now I'll calculate the same three numbers **without pandas**, and without Python's built-in `statistics` module. That means reading the file and doing the math myself.

This section doesn't touch any of the pandas variables from Part 1. It reads the same CSV file from scratch, so it would still work if you copied it into a new notebook. That's why it repeats the import and the file name.

### Reading the file

Python's built-in [`csv`](https://docs.python.org/3/library/csv.html) module reads the file one row at a time. Each row comes back as a dictionary like `{"tree_dbh": "3", "status": "Alive", ...}`. Every value is text, so I convert each diameter to a whole number with `int()`. I only keep living trees, to match Part 1.

In [10]:
import csv

DATA_FILE = "street_trees_2015.csv"

diameters = []
with open(DATA_FILE, newline="") as file:
    for row in csv.DictReader(file):
        if row["status"] == "Alive":
            diameters.append(int(row["tree_dbh"]))

print(f"Read {len(diameters):,} living trees.")

Read 652,173 living trees.


### Mean

The mean is the total of all the diameters divided by the number of trees.

In [11]:
mean_hard = sum(diameters) / len(diameters)

print(f"Mean: {mean_hard:.2f} inches")

Mean: 11.71 inches


### Median

To find the median, I sort the diameters from smallest to largest and take the one in the middle. If there's an even number of trees, there are *two* middle values, so I average them.

In [12]:
sorted_diameters = sorted(diameters)
tree_count = len(sorted_diameters)
middle = tree_count // 2

if tree_count % 2 == 1:
    median_hard = sorted_diameters[middle]
else:
    median_hard = (sorted_diameters[middle - 1] + sorted_diameters[middle]) / 2

print(f"Median: {median_hard:g} inches")

Median: 10 inches


### Mode

For the mode, I use a dictionary to tally how many trees have each diameter, shaped like `{diameter: number of trees, ...}`. Then I find the highest tally and collect every diameter that has it. Like pandas, this can return more than one mode if there's a tie.

In [13]:
counts_by_diameter = {}
for diameter in diameters:
    counts_by_diameter[diameter] = counts_by_diameter.get(diameter, 0) + 1

highest_count = max(counts_by_diameter.values())
modes_hard = sorted(
    diameter
    for diameter, count in counts_by_diameter.items()
    if count == highest_count
)

print(f"Mode: {', '.join(str(mode) for mode in modes_hard)} inches")
print(f"({highest_count:,} living trees have that diameter.)")

Mode: 4 inches
(58,149 living trees have that diameter.)


### Did both methods agree?

As a final check, this cell compares the Part 1 and Part 2 results. It's the only cell that uses both. Computers store decimals with tiny rounding errors, so for the mean I check that the two values are *extremely close* (with [`math.isclose()`](https://docs.python.org/3/library/math.html#math.isclose)) instead of exactly equal.

In [14]:
print("Mean matches:  ", math.isclose(mean_pandas, mean_hard))
print("Median matches:", median_pandas == median_hard)
print("Mode matches:  ", modes_pandas == modes_hard)

Mean matches:   True
Median matches: True
Mode matches:   True


They match. pandas did the same work in one line per statistic, but at least now I know what it's doing behind the scenes.

## Part 3: Drawing the trees

Three numbers only tell part of the story, so I want to see the **shape** of the data: how many trees there are at each trunk size. The rule for this project is that pandas can do the counting, but the drawing has to use plain Python. No plotting libraries allowed.

My plan:

1. Group the trees into 3-inch-wide size ranges (0–2 inches, 3–5 inches, and so on), one row of the chart per range.
2. A few trees are extremely thick, and drawing a row for every range up to 400+ inches would make the chart enormous. So I lump the thickest 1% of trees into a single final "and up" row.
3. Draw each row as a line of 🌳 emoji, where each emoji stands for a fixed number of trees.

### Counting (with pandas)

In [15]:
BIN_WIDTH = 3  # inches covered by each row of the chart

# Everything above the 99th percentile goes into one final "and up" row.
# Round the cutoff up to a multiple of BIN_WIDTH so all other rows are the same width.
tail_cutoff = math.ceil(living_diameters.quantile(0.99) / BIN_WIDTH) * BIN_WIDTH

bin_starts = living_diameters.clip(upper=tail_cutoff) // BIN_WIDTH * BIN_WIDTH
bin_counts = (
    bin_starts.value_counts()
    .reindex(range(0, tail_cutoff + 1, BIN_WIDTH), fill_value=0)
    .sort_index()
)

# Hand the drawing code a plain Python dictionary: {bin start: number of trees}
trees_per_bin = {int(start): int(count) for start, count in bin_counts.items()}
trees_per_bin

{0: 41322,
 3: 150485,
 6: 96627,
 9: 95424,
 12: 71522,
 15: 52808,
 18: 43136,
 21: 29936,
 24: 25213,
 27: 17309,
 30: 13012,
 33: 7181,
 36: 4190,
 39: 4008}

### Drawing (with plain Python)

A few decisions:

- **Scale:** The longest bar is limited to about 20 emoji so the chart fits on a small laptop screen. Then I round the "trees per emoji" up to a friendly number (1, 2, or 5 followed by zeros) so the legend is easy to read.
- **Tiny rows:** A row with fewer trees than half an emoji gets a `·`, so it's clear the row isn't empty.
- **Landmarks:** I mark the rows that contain the median, mean, and mode from Part 1, so you can see where those numbers fall on the chart.

In [16]:
MAX_ICONS = 20
ICON = "🌳"
TINY_MARK = "·"


def round_up_to_nice_number(value):
    # Round up to 1, 2, or 5 times a power of 10, e.g. 7,312 -> 10,000
    power_of_ten = 10 ** math.floor(math.log10(value))
    for multiplier in (1, 2, 5, 10):
        if value <= multiplier * power_of_ten:
            return multiplier * power_of_ten


def bin_label(start):
    if start == tail_cutoff:
        return f"{start}+ in"
    return f"{start}–{start + BIN_WIDTH - 1} in"


def bin_start_for(value):
    return int(min(value, tail_cutoff) // BIN_WIDTH * BIN_WIDTH)


trees_per_icon = round_up_to_nice_number(max(trees_per_bin.values()) / MAX_ICONS)

# Which rows should get a "◀ median" style marker?
landmarks = {}
for name, value in [
    ("median", median_hard),
    ("mean", mean_hard),
    ("mode", modes_hard[0]),
]:
    landmarks.setdefault(bin_start_for(value), []).append(name)

label_width = max(len(bin_label(start)) for start in trees_per_bin)

print("How Thick Are NYC's Living Street Trees?")
print("Trunk diameter at breast height, 2015 Street Tree Census")
print(f"Each {ICON} = {trees_per_icon:,} trees   {TINY_MARK} = fewer than {trees_per_icon // 2:,}")
print()
print(f"{'Diameter':>{label_width}}   Number of trees")

for start, count in trees_per_bin.items():
    icon_count = round(count / trees_per_icon)
    if icon_count > 0:
        bar = ICON * icon_count
    elif count > 0:
        bar = TINY_MARK
    else:
        bar = ""

    line = f"{bin_label(start):>{label_width}} │ {bar} {count:,}"
    if start in landmarks:
        line += "  ◀ " + " & ".join(landmarks[start])
    print(line)

How Thick Are NYC's Living Street Trees?
Trunk diameter at breast height, 2015 Street Tree Census
Each 🌳 = 10,000 trees   · = fewer than 5,000

Diameter   Number of trees
  0–2 in │ 🌳🌳🌳🌳 41,322
  3–5 in │ 🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳 150,485  ◀ mode
  6–8 in │ 🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳 96,627
 9–11 in │ 🌳🌳🌳🌳🌳🌳🌳🌳🌳🌳 95,424  ◀ median & mean
12–14 in │ 🌳🌳🌳🌳🌳🌳🌳 71,522
15–17 in │ 🌳🌳🌳🌳🌳 52,808
18–20 in │ 🌳🌳🌳🌳 43,136
21–23 in │ 🌳🌳🌳 29,936
24–26 in │ 🌳🌳🌳 25,213
27–29 in │ 🌳🌳 17,309
30–32 in │ 🌳 13,012
33–35 in │ 🌳 7,181
36–38 in │ · 4,190
  39+ in │ · 4,008


**What the chart shows:** the trees are piled up at the thin end, with a long, thinning tail of bigger trees. The most crowded row has the young 3-to-5-inch trees. After that, every 3 inches of extra thickness means fewer trees, and a tree more than 3 feet across is rare.

This shape explains the gap from Part 1. The mode sits in the tallest bar, the median a little to the right, and the long tail of giants (plus a few typos) pulls the mean to the right of the median. When data is lopsided like this, the median is usually the best single number for "typical."

## Caveats

The project is about practicing the mechanics, not publishing science, but a few things are worth knowing:

- **The data is from 2015.** Every tree has had about a decade to grow, and some have been planted or removed since. A newer census would give different numbers.
- **Volunteers did a lot of the measuring.** NYC Parks trained them, but measurement errors happen, as the 400-inch "trees" show. I kept those obvious outliers rather than picking an arbitrary cutoff.
- **A few living trees have a diameter of 0.** These might be brand-new saplings too small to measure at breast height, or they might be mistakes. There are only a couple hundred, out of more than 650,000 living trees, so they barely affect the results, and I kept them.
- **Diameters are whole inches.** Rounding to the nearest inch is what makes a meaningful mode possible. If diameters were recorded as precise decimals, almost no two trees would match exactly.
- **Only street trees are counted.** Trees in parks, backyards, and forests (including some of the city's biggest and oldest) aren't in this dataset, so this is about the trees along city sidewalks, not every tree in NYC.
- **The download asks for at most 1,000,000 rows.** That's more than the dataset has today, but if it ever grew past that, the `$limit` in the URL would need to go up.

## Takeaways

- A "typical" living NYC street tree is roughly **10 inches thick**, about the width of a dinner plate. That's the median, and the mean is in the same neighborhood.
- The **most common** street tree is much skinnier than that. NYC's street trees skew young.
- Stumps are stored with a diameter of 0. If I hadn't checked those zeros, they would have quietly dragged every number down. Investigating odd values was the most useful step in the whole analysis.
- pandas and plain Python gave identical answers. pandas was much shorter to write, but doing it by hand showed exactly what "median" and "mode" mean.

## Sources

- NYC Department of Parks & Recreation, [2015 Street Tree Census - Tree Data](https://data.cityofnewyork.us/Environment/2015-Street-Tree-Census-Tree-Data/uvpi-gqnh), NYC Open Data (data dictionary under *About → Attachments*)
- NYC Parks, [TreesCount! Street Tree Census](https://www.nycgovparks.org/trees/treescount)